# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's (page's) performance record on a single day
(report_date). Each row is uniquely identified by the triple
(client_hash_id, content_hash_id, report_date) — the same page appears in
multiple rows, one per day it has data for.

Time window: a mid-panel month slice of fact_content_daily_performance,
month=2026-03 (March 2026) — report_date is restricted to that partition,
not the final month (2026-06), which is kept sealed as a held-out test
month to avoid label leakage.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [1]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

In [2]:
field_classification = {
    "feature":  ["imp_trailing", "clk_trailing", "pos_trailing",
                 "has_ga4_data", "content_age_days"],
    "label":    ["is_declining"],
    "context":  ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["trend_direction", "trend_pct", "ga4_* (when ga4_data_available IS FALSE)"],
}
for bucket, cols in field_classification.items():
    print(f"{bucket:10} -> {cols}")

feature    -> ['imp_trailing', 'clk_trailing', 'pos_trailing', 'has_ga4_data', 'content_age_days']
label      -> ['is_declining']
context    -> ['client_hash_id', 'content_hash_id', 'report_date']
excluded   -> ['trend_direction', 'trend_pct', 'ga4_* (when ga4_data_available IS FALSE)']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Tekrar eden satır sayısı:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tekrar eden satır sayısı: 0


,client_hash_id,content_hash_id,report_date,c


In [4]:
counts = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date,
        COUNT(DISTINCT client_hash_id)   AS n_clients,
        COUNT(DISTINCT content_hash_id)  AS n_content
    FROM read_parquet('{MONTH_PATH}')
""").df()
counts

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [5]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{MONTH_PATH}')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


In [6]:
DECISION_DAY = '2026-03-15'

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') dc
      ON fx.content_hash_id = dc.content_hash_id
    WHERE dc.content_created_date <= DATE '{DECISION_DAY}'
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_trailing,clk_trailing,pos_trailing,has_ga4_data,content_age_days
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,5.222776,0,31
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,4.638889,0,31
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,3.737399,0,31
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,3.597222,0,31
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,6.156643,0,31


Five features:
1. `imp_trailing` — sum of impressions up to March 15; only past days, nothing after.
2. `clk_trailing` — same reasoning, only days on or before the decision day.
3. `pos_trailing` — average search position from days already observed.
4. `has_ga4_data` — whether GA4 tracking existed as of that date, a fact of that moment.
5. `content_age_days` — computed from `content_created_date`, which is fixed and
   always at or before the decision day (rows where the page was created AFTER
   March 15 were excluded entirely).

In [7]:
labels = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)
print(merged['is_declining'].value_counts())

is_declining
0    268443
1     49673
Name: count, dtype: int64


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leaky = merged[['imp_trailing', 'clk_trailing', 'pos_trailing', 'has_ga4_data',
                   'content_age_days', 'imp_after']].fillna(0)
y = merged['is_declining']

model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leak'li AUC: {auc_leaky:.3f}")

Leak'li AUC: 1.000


In [9]:
X_honest = merged[['imp_trailing', 'clk_trailing', 'pos_trailing',
                    'has_ga4_data', 'content_age_days']].fillna(0)

model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
auc_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"Honest AUC: {auc_honest:.3f}")

Honest AUC: 0.722


**The trap:** Adding `imp_after` (the column the label is directly computed from) as
a feature pushed AUC to 1.000 — the model was effectively seeing the answer. Removing
it dropped AUC to a realistic 0.722, consistent with the 0.750 AUC seen in notebook 02.
This confirms label-derived columns must never enter the feature set, even when they
look like ordinary numeric columns.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
dim_clients_cols = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/dim_clients.parquet') LIMIT 1
""").df()
dim_clients_cols

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [11]:
partial_history = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
    WHERE gsc_data_start > DATE '2026-03-01'
       OR ga4_data_start > DATE '2026-03-01'
""").df()
print(f"Mart 2026 başlamadan önce tam geçmişi olmayan client sayısı: {len(partial_history)}")
partial_history.head()

Mart 2026 başlamadan önce tam geçmişi olmayan client sayısı: 27


,client_hash_id,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,NaT,2026-05-22
1,client_06d356715a8ff3b6,2026-04-10,2026-04-06
2,client_0b245132bb722950,2026-04-12,2026-04-24
3,client_157ffe4d4a595515,2026-02-19,2026-03-09
4,client_1a730cb2640a1abf,2026-02-19,2026-03-05


**Result and a correction, not just a softening:** the pattern itself is real and fairly common:
62.2% (704 of 1,132) of baseline-flagged rows were pages the rule called risky but that stayed
stable, and the model's top-50 correctly excluded them. That part of the original claim holds up at
scale, better than 3 examples could prove on their own.

But the *mechanism* in the original sentence doesn't survive the larger check. The original claim
was built on 3 examples of **old** pages (97-209 days) and explained the pattern as "the model also
weighs age, which an old, age-blind rule misses." Across the full 704-row pattern, the median
`content_age_days` is **54 days** young content, the opposite of what the 3-example story
described. Whatever the model is actually picking up on for most of these 704 rows, it is probably
not "this content is old but I know old-and-refreshed content is fine" the 3 examples I originally
picked were not representative of the pattern's typical case.

**Rewritten claim (final):** "In this held-out, client-grouped test set, the model's top-50 excludes
62% of the pages the position+CTR rule flags as risky, and those excluded pages mostly stay stable
(median age 54 days, not the older profile my Week-5 write-up described from 3 examples). This is an
observed, decision-support pattern the earlier explanation of *why* age helps was itself an
overclaim from too small a sample, and should be dropped until re-examined with feature importance
or SHAP-style attribution rather than another handful of examples."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.